In [1]:
%matplotlib inline

In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.8 MB/s eta 0:00:00


In [3]:
from ultralytics import YOLO
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import os
from IPython.display import display, clear_output
import ipywidgets as widgets
import torch
import time
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import img_to_array
from sklearn.preprocessing import LabelEncoder
import numpy as np

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [6]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'


In [4]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [7]:
dataset_path = "/content/drive/MyDrive/dataset/FDDB/data.yaml"
model_path = "/content/drive/MyDrive/FaceDetectModel/train5/weights/best.pt"

# Kiểm tra nếu model đã tồn tại
if os.path.exists(model_path):
    print("Load model đã tạo...")
    model = YOLO(model_path)  # Load mô hình đã huấn luyện
else:
    print("Huấn luyện model...")
    model = YOLO("yolov8n.pt")  # Load model pre-trained
    results = model.train(
        data=dataset_path,
        epochs=50,               # Giảm số epoch để huấn luyện nhanh hơn
        imgsz=640,               # Giảm kích thước ảnh
        batch=8,
        verbose=False,
        project="/content/drive/MyDrive/FaceDetectModel",
        save_period=1
    )

Huấn luyện model...
Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/dataset/FDDB/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train5, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_

In [12]:
#Kiểm tra mô hình sau huấn luyện
model = YOLO("/content/drive/MyDrive/FaceDetectModel/train5/weights/best.pt")  # Load mô hình tốt nhất
results = model("https://ultralytics.com/images/zidane.jpg", save=True)  # Test trên một ảnh mẫu


image 1/1 /content/zidane.jpg: 384x640 2 faces, 45.1ms
Speed: 2.0ms preprocess, 45.1ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)
Results saved to /content/runs/detect/predict


In [ ]:
def process_frame_emotion(frame, yolo_model, cnn_model):
    # Load label encoder và emotion_labels
    import pickle
    with open("models/label_encoder.pkl", "rb") as f:
        le = pickle.load(f)
    with open("models/emotion_labels.pkl", "rb") as f:
        emotion_labels = pickle.load(f)

    # Sau đó bạn có thể dùng emotion_labels trong predict:
    #emotion_pred = model.predict(face_array)
    #emotion_label = emotion_labels[np.argmax(emotion_pred)]

    results = yolo_model(frame, verbose=False)[0]
    for box in results.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        conf = box.conf[0].item()

        # Cắt khuôn mặt từ ảnh
        face = frame[y1:y2, x1:x2]
        if face.size == 0:
            continue

        # Tiền xử lý ảnh khuôn mặt cho CNN
        face_input_size = (48, 48)
        face_resized = cv2.resize(face, (80, 80))  # width, height
        face_array = img_to_array(face_resized)
        face_array = face_array.astype("float32") / 255.0
        face_array = np.expand_dims(face_array, axis=0)

        # Dự đoán cảm xúc
        emotion_pred = cnn_model.predict(face_array, verbose=0)
        emotion_label = emotion_labels[np.argmax(emotion_pred)]
        emotion_conf = np.max(emotion_pred)

        # Hiển thị bounding box và nhãn
        label_text = f"{emotion_label}: {emotion_conf:.1%}"
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, label_text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX,
                    0.8, (0, 255, 0), 2, cv2.LINE_AA)

    return frame

In [13]:
# ============================
# Load mô hình
# ============================
print("[INFO] Đang tải model YOLO và CNN...")
yolo_model = YOLO("/content/drive/MyDrive/FaceDetectModel/train5/weights/best.pt")  # Model YOLO để phát hiện khuôn mặt
cnn_model = load_model("models/model_emotion.h5")  # Model CNN để phân loại cảm xúc

# ============================
# Khởi động camera và chạy mô hình
# ============================
print("[INFO] Mở webcam...")
webcamera = cv2.VideoCapture(0)

while True:
    success, frame = webcamera.read()
    if not success:
        break

    processed_frame = process_frame_emotion(frame, yolo_model, cnn_model)

    cv2.imshow("Emotion Detection", processed_frame)

    if cv2.waitKey(1) == ord('q'):
        break

webcamera.release()
cv2.destroyAllWindows()



[INFO] Đang tải model YOLO và CNN...


FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'models/model_emotion.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)